# SVM Notebook

Starter notebook for the Support Vector Machine workflow.

This notebook is intentionally lightweight so the team can expand it with the final analysis, visuals, and presentation notes.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

def find_ml_root(start=None):
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "cleaned_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate ml/data/processed/cleaned_dataset.csv")

ML_DIR = find_ml_root()
DATA_PATH = ML_DIR / "data" / "processed" / "cleaned_dataset.csv"
MODEL_PATH = ML_DIR / "models" / "import_risk_svm.joblib"
METRICS_PATH = ML_DIR / "models" / "svm_metrics.json"
FEATURE_COLUMNS = [
    "price_usd",
    "weight_kg",
    "volume_m3",
    "max_dimension_m",
    "dimension_sum_m",
    "density_kg_m3",
    "value_per_kg",
    "value_per_m3",
]
TARGET_COLUMN = "risk"
RANDOM_STATE = 42
MAX_ROWS_FOR_SVM = 80000

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

print(df.shape)
display(df.head())
display(df[TARGET_COLUMN].value_counts())

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()

ax = class_counts.plot(kind="bar", color=["#2563eb", "#ef4444"])
ax.set_title("Class Distribution")
ax.set_xlabel("Risk Class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
sample_df = df.sample(n=min(len(df), MAX_ROWS_FOR_SVM), random_state=RANDOM_STATE)
X = sample_df[FEATURE_COLUMNS]
y = sample_df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

svm_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LinearSVC(C=1.0, class_weight="balanced", dual=False, max_iter=5000, random_state=RANDOM_STATE),
        ),
    ]
)

param_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
    "classifier__class_weight": [None, "balanced"],
}

search = GridSearchCV(
    svm_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=1,
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, pos_label="HIGH RISK", zero_division=0),
    "recall": recall_score(y_test, y_pred, pos_label="HIGH RISK", zero_division=0),
    "f1": f1_score(y_test, y_pred, pos_label="HIGH RISK", zero_division=0),
}

print("Best params:", search.best_params_)
print(metrics)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["LOW RISK", "HIGH RISK"])

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["LOW RISK", "HIGH RISK"],
    yticklabels=["LOW RISK", "HIGH RISK"],
)
plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
metric_series = pd.Series({
    "Accuracy": metrics["accuracy"],
    "Precision": metrics["precision"],
    "Recall": metrics["recall"],
    "F1 Score": metrics["f1"],
})

ax = metric_series.plot(kind="bar", color="#0f766e")
ax.set_ylim(0, 1)
ax.set_title("SVM Metric Summary")
ax.set_ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
decision_scores = best_model.decision_function(X_test)
score_frame = pd.DataFrame({"score": decision_scores, "risk": y_test.values})

sns.histplot(data=score_frame, x="score", hue="risk", element="step", stat="density", common_norm=False)
plt.title("SVM Decision Score Distribution")
plt.tight_layout()
plt.show()